In [7]:
%pip install neo4j datasets google-genai pydantic python-dotenv numpy tqdm --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import pickle
import sys
from pathlib import Path

from datasets import load_dataset
from dotenv import load_dotenv

load_dotenv("../.env")

sys.path.insert(0, str(Path("..").resolve()))

In [2]:
from graph import KnowledgeGraphLoader

In [3]:
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "password123")

In [4]:
raw_dataset = load_dataset("dgslibisey/MuSiQue", split="validation")
print(f"Total validation examples: {len(raw_dataset):,}")

answerable_dataset = raw_dataset.filter(lambda x: x["answerable"])
print(f"Answerable examples: {len(answerable_dataset):,}")

dataset = answerable_dataset.select(range(1000))
print(f"Selected {len(dataset)} examples")

Total validation examples: 2,417
Answerable examples: 2,417
Selected 1000 examples


In [5]:
def transform_musique_to_loader_format(example):
    """Transform MuSiQue format to match loader expectations.

    Output format per example::

        {
            "question": str,
            "answer": str,
            "paragraphs": [{"title": str, "text": str, "is_supporting": bool}, ...],
        }
    """
    return {
        "question": example["question"],
        "answer": example["answer"],
        "paragraphs": [
            {
                "title": para["title"],
                "text": para["paragraph_text"],
                "is_supporting": para["is_supporting"],
            }
            for para in example["paragraphs"]
        ],
    }

transformed_dataset = [transform_musique_to_loader_format(example) for example in dataset]
print(f"Transformed {len(transformed_dataset)} examples")
print(f"Q: {transformed_dataset[0]['question']}")
print(f"A: {transformed_dataset[0]['answer']}")
print(f"Paragraphs: {len(transformed_dataset[0]['paragraphs'])}")

Transformed 1000 examples
Q: Who is the spouse of the Green performer?
A: Miquette Giraudy
Paragraphs: 20


In [6]:
loader = KnowledgeGraphLoader(
    neo4j_uri=NEO4J_URI,
    neo4j_user=NEO4J_USER,
    neo4j_password=NEO4J_PASSWORD,
    cache_dir=Path("cache"),
)

[Embedder] Loaded 828 cached embeddings
[Embedder] Loaded 59347 cached embeddings
[Loader] Loaded 10988 cached extractions


In [7]:
loader.clear_and_init()

Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL (n) { ... }', position=<SummaryInputPosition line=3, column=17, offset=43>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 43, 'line': 3, 'column': 17}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n                MATCH (n)\n                CALL {\n                    WITH n\n                    DETACH DELETE n\n                } IN TRANSACTIONS OF 10000 ROWS\n                '


[Neo4j] Database cleared
[Neo4j] Indexes created


In [8]:
ground_truth = loader.load_examples(
    transformed_dataset,
    max_extraction_workers=30,
    max_embedding_workers=5,
    save_every=1000,
)

[Loader] Total unique paragraphs: 12493
[Loader] Total unique documents (titles): 10785

[Loader] Phase 1: Extracting entities and relations...
[Loader] Need to extract 0 (10988 cached)

[Loader] Phase 2: Loading into Neo4j...


Loading Neo4j: 100%|██████████| 1000/1000 [25:38<00:00,  1.54s/it] 



[Loader] Phase 3: Post-processing entities...
[Loader] Need to embed 53590 entity descriptions
[Embedder] Need 0 embeddings (59347 cached)


In [9]:
loader.finalize()

[Neo4j] Paragraph vector index created
[Neo4j] Entity vector index created


In [10]:
with open("ground_truth-musique.pkl", "wb") as f:
    pickle.dump(ground_truth, f)

print(f"Saved ground truth for {len(ground_truth)} questions")

Saved ground truth for 1000 questions


In [11]:
stats = loader.get_stats()
for key, value in stats.items():
    print(f"{key}: {value}")

documents: 10785
paragraphs: 12493
entities: 55232
has_paragraph: 12493
mentions: 96865
relations: 75768


In [ ]:
loader.close()